1. Install Library yang Dibutuhkan

In [1]:
!pip install pymupdf pandas

   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
    --------------------------------------- 0.3/19.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/19.2 MB 881.1 kB/s eta 0:00:22
   - -------------------------------------- 0.8/19.2 MB 995.0 kB/s eta 0:00:19
   -- ------------------------------------- 1.0/19.2 MB 995.2 kB/s eta 0:00:19
   -- ------------------------------------- 1.3/19.2 MB 1.0 MB/s eta 0:00:18
   --- ------------------------------------ 1.6/19.2 MB 1.0 MB/s eta 0:00:17
   --- ------------------------------------ 1.8/19.2 MB 1.1 MB/s eta 0:00:17
   ---- ----------------------------------- 2.1/19.2 MB 1.1 MB/s eta 0:00:16
   ---- ----------------------------------- 2.1/19.2 MB 1.1 MB/s eta 0:00:16
   ----- -------------------

Cell 1: Poin (i, ii, iii) - Ekstraksi dan Pembersihan

In [ ]:
import os
import fitz  
import re

# i. Seleksi & Unduh (Sudah dilakukan, file ada di /data/raw)
folder_raw = "../data/raw"
data_kasus = [] # List untuk menyimpan data sementara ke tahap validasi

def bersihkan_teks(teks):
    # iii.2. Normalisasi spasi dan karakter (lower-case)
    teks_bersih = re.sub(r'\s+', ' ', teks)
    teks_bersih = teks_bersih.lower()
    return teks_bersih.strip()

print("Memulai Tahap (ii) & (iii): Ekstraksi dan Pembersihan...")

for nama_file in os.listdir(folder_raw):
    if nama_file.endswith('.pdf'):
        path_pdf = os.path.join(folder_raw, nama_file)
        # Menyiapkan nama file .txt
        nama_file_txt = nama_file.replace('.pdf', '.txt')
        path_txt = os.path.join(folder_raw, nama_file_txt)
        
        # ii. Konversi & Ekstraksi Teks (Ubah PDF -> plain text)
        doc = fitz.open(path_pdf)
        teks_asli = ""
        for page in doc:
            teks_asli += page.get_text()
            
        # iii.1. & iii.2. Pembersihan teks
        teks_bersih = bersihkan_teks(teks_asli)
        
        # iii.3. Simpan hasil di folder /data/raw/*.txt
        with open(path_txt, 'w', encoding='utf-8') as f_txt:
            f_txt.write(teks_bersih)
            
        # Simpan informasi ke memori untuk digunakan pada Poin (iv) Validasi
        data_kasus.append({
            'file_name': nama_file_txt,
            'panjang_raw': len(teks_asli),
            'panjang_clean': len(teks_bersih),
            'teks_bersih': teks_bersih
        })

print(f"Selesai! {len(data_kasus)} file .txt berhasil dibuat di folder {folder_raw}")

Memulai Tahap (ii) & (iii): Ekstraksi dan Pembersihan...
Selesai! 30 file .txt berhasil dibuat di folder ../data/raw


Cell 2: Poin (iv, v) - Validasi dan Output (Logging)

In [12]:
import logging
import pandas as pd

# v. Output: Menyiapkan folder logs
folder_logs = "../logs"
os.makedirs(folder_logs, exist_ok=True)
path_log = os.path.join(folder_logs, "cleaning.log")

# Reset log agar tidak menumpuk kalau di-run berkali-kali
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    filename=path_log, 
    level=logging.INFO, 
    format='%(asctime)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

print("Memulai Tahap (iv): Validasi Keutuhan Teks...")
total_valid = 0
total_invalid = 0

for kasus in data_kasus:
    # iv.1. Periksa keutuhan teks (minimal 80% isi putusan tersedia)
    if kasus['panjang_raw'] == 0:
        rasio = 0
    else:
        rasio = (kasus['panjang_clean'] / kasus['panjang_raw']) * 100
        
    if rasio >= 80:
        status = "VALID"
        total_valid += 1
    else:
        status = "INVALID"
        total_invalid += 1
        
    # iv.2. Catat log file pembersihan
    pesan_log = f"File: {kasus['file_name']} | Raw: {kasus['panjang_raw']} chars | Clean: {kasus['panjang_clean']} chars | Keutuhan: {rasio:.2f}% | Status: {status}"
    logging.info(pesan_log)

# Mengubah data_kasus menjadi DataFrame 'df' agar siap dilanjutkan ke Tahap 2 (Ekstraksi Metadata)
df = pd.DataFrame(data_kasus)

print(f"Validasi Selesai!")
print(f"- Total Valid (>= 80%): {total_valid}")
print(f"- Total Invalid (< 80%): {total_invalid}")
print(f"v.2. Log history pembersihan berhasil disimpan di: {path_log}")

Memulai Tahap (iv): Validasi Keutuhan Teks...
Validasi Selesai!
- Total Valid (>= 80%): 30
- Total Invalid (< 80%): 0
v.2. Log history pembersihan berhasil disimpan di: ../logs\cleaning.log
